# Practical Machine Learning with TensorFlow — NLP & Transformers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/ml-course-notebooks/blob/main/tensorflow_course.ipynb)

Code from all 28 episodes of the TensorFlow video series, following the pipeline spine **Collect → Preprocess → Build → Train → Evaluate → Save → Deploy → Predict**, with a heavy NLP / Transformer track.

> Each cell holds one episode's code as shown in the video. Some episodes are multi-stage (modular blocks concatenated); the NLP/Transformer episodes (TF11–TF20) build toward a transformer from scratch.

In [ ]:
!pip -q install tensorflow-datasets  # tensorflow + keras are preinstalled on Colab

## TF01 · What Is TensorFlow? Tensors, Keras, and the Map We'll Follow

In [ ]:
# 01_hello_tensorflow.py — your first tensors + proof Keras is one import away
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers   # TF 2.16+ ships Keras 3; `from keras import layers` also works
print("TensorFlow version:", tf.__version__)

# --- Make a few tensors by hand ---
scalar = tf.constant(7)
vector = tf.constant([5, 3, 9])
matrix = tf.constant([[1, 2, 3], [4, 5, 6]])

# --- Read the label: shape + dtype ---
print("scalar:", scalar.shape, scalar.dtype)
print("vector:", vector.shape, vector.dtype)
print("matrix:", matrix.shape, matrix.dtype)

# --- Real math on tensors ---
a = tf.constant([[1, 2], [3, 4]])
b = tf.constant([[5, 6], [7, 8]])
print("a + b =\n", (a + b).numpy())          # element-wise add
print("a @ b =\n", (a @ b).numpy())          # matrix multiply
print("reduce_mean =", tf.reduce_mean(tf.cast(tf.range(1, 7), tf.float32)).numpy())

model = keras.Sequential([layers.Dense(1)]); print("Keras model ready:", model)

## TF02 · The Whole Loop in One Sitting: Your First End-to-End Model

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# 1 · COLLECT — Fashion-MNIST ships with Keras
fashion = tf.keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = fashion.load_data()

# 2 · PREPROCESS — scale pixels from 0–255 into 0–1
x_train = x_train / 255.0
x_test = x_test / 255.0
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# 3 · BUILD — Input, flatten the grid, two Dense layers, 10 logits out
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10),
])

# 4 · COMPILE — Adam + logits loss + accuracy metric
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"])

# 5 · TRAIN — fit() is the loop; hold out 10% to validate
model.fit(x_train, y_train, epochs=8, validation_split=0.1)

# 6 · EVALUATE — the only honest score is on unseen test data
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"\nTest accuracy: {test_acc:.4f}")

# 7 · SAVE + DEPLOY — one file holds architecture + weights
model.save("fashion.keras")
reloaded = tf.keras.models.load_model("fashion.keras")

# 8 · PREDICT — one image -> 10 logits -> softmax -> argmax
img = x_test[0]
logits = reloaded.predict(np.expand_dims(img, axis=0))[0]
probs = tf.nn.softmax(logits).numpy()
pred = int(np.argmax(probs))
print(f"Predicted: {class_names[pred]}  ({probs[pred]:.2f})")

# draw the white result panel: thumbnail + probability bars
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(img, cmap="gray"); ax1.axis("off")
ax1.set_title(class_names[pred])
bars = ax2.bar(range(10), probs, color="#c9c9c9")
bars[pred].set_color("#ffd483")
ax2.set_xticks(range(10)); ax2.set_xticklabels(class_names, rotation=90)
plt.tight_layout(); plt.savefig("prediction.png", dpi=130)

## TF03 · Feeding the Beast: tf.data Pipelines That Don't Starve Your GPU

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

AUTOTUNE = tf.data.AUTOTUNE

# --- toy in-memory data ---
features = np.random.rand(256, 4).astype("float32")
labels = np.random.randint(0, 2, size=(256,)).astype("int32")
ds = tf.data.Dataset.from_tensor_slices((features, labels))

# --- per-element preprocessing: scale each row by its own max ---
def normalize(x, y):
    return x / tf.reduce_max(x), y

ds = ds.map(normalize, num_parallel_calls=AUTOTUNE)

# --- cache -> shuffle -> batch -> prefetch ---
ds = ds.cache().shuffle(1000).batch(32).prefetch(AUTOTUNE)
print(ds)

# --- pull exactly one batch and check shapes ---
for xb, yb in ds.take(1):
    print("features batch:", xb.shape)
    print("labels   batch:", yb.shape)

# --- real data: a folder of image files ---
train_ds = keras.utils.image_dataset_from_directory(
    "flowers",
    image_size=(180, 180),
    batch_size=32,
    label_mode="int",
)
class_names = train_ds.class_names
print("classes:", class_names)

# --- normalize raw 0-255 pixels to [0, 1] ---
rescale = keras.layers.Rescaling(1.0 / 255)
train_ds = train_ds.map(
    lambda x, y: (rescale(x), y),
    num_parallel_calls=AUTOTUNE,
)

# --- cache + prefetch for speed ---
train_ds = train_ds.cache().prefetch(AUTOTUNE)

# --- preview one batch ---
for images, lbls in train_ds.take(1):
    print("image batch:", images.shape)
    print("label batch:", lbls.shape)

## TF04 · The Neuron, the Layer, the Network: Building an ANN by Hand

In [ ]:
# 04_ann_build.py
# Build (deep): a feedforward ANN that classifies irises into 3 species.
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Load & preprocess the tabular data (4 features, 3 classes) ---
data = load_iris()
X, y = data.data.astype("float32"), data.target.astype("int32")
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype("float32")
X_val = scaler.transform(X_val).astype("float32")

# --- Build the network with the functional API ---
inputs = keras.Input(shape=(4,), name="features")
x = layers.Dense(64, activation="relu", name="hidden_1")(inputs)
x = layers.Dense(32, activation="relu", name="hidden_2")(x)
outputs = layers.Dense(3, activation="softmax", name="species")(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="iris_ann")


# --- Read the blueprint: layers + parameter counts ---
model.summary()

# --- Compile: how the model should learn ---
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

# --- Train and capture the history ---
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=16, verbose=2)

# --- Plot train vs. validation accuracy ---
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
epochs_range = range(1, len(acc) + 1)
plt.figure(figsize=(6, 6))
plt.plot(epochs_range, acc, label="train accuracy")
plt.plot(epochs_range, val_acc, label="val accuracy")
plt.xlabel("epoch"); plt.ylabel("accuracy")
plt.title("Training vs. validation accuracy")
plt.legend(); plt.tight_layout()
plt.savefig("ann_accuracy.png", dpi=150)
plt.show()

## TF05 · How It Actually Learns: Loss, Gradients, and Backprop

In [ ]:
# 05_training_loop.py
# Same ANN, two ways to train it: a hand-written GradientTape loop
# and the one-line model.fit() that wraps exactly the same steps.
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_iris
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

# ---- Data: 4 flower measurements -> 3 species (logits) ----
iris = load_iris()
X = tf.constant(iris.data, dtype=tf.float32)        # (150, 4)
y = tf.constant(iris.target, dtype=tf.int32)        # (150,)

def build_model():
    return keras.Sequential([
        keras.layers.Input(shape=(4,)),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dense(3),                      # raw logits, no softmax
    ])

model = build_model()

# ---- The two ingredients of learning ----
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = keras.optimizers.SGD(learning_rate=0.1)

# ---- Hand-written gradient-descent loop ----
for step in range(501):
    with tf.GradientTape() as tape:                 # start recording
        logits = model(X, training=True)            # forward pass
        loss = loss_fn(y, logits)                   # one number: how wrong
    grads = tape.gradient(loss, model.trainable_variables)   # backprop
    optimizer.apply_gradients(zip(grads, model.trainable_variables))  # w := w - lr*grad
    if step % 50 == 0:
        print(f"step {step:3d}  loss {loss.numpy():.4f}")

print("\nfit() did the same in 1 line:")

# ---- The exact same loop, hidden inside model.fit() ----
model2 = build_model()
model2.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1),
               loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
               metrics=["accuracy"])
hist = model2.fit(X, y, epochs=500, batch_size=150, verbose=0)
print(f"fit() final loss {hist.history['loss'][-1]:.4f}")

# ---- Compare three learning rates ----
curves = {}
for lr in [0.001, 0.1, 1.0]:
    m = build_model()
    opt = keras.optimizers.SGD(learning_rate=lr)
    losses = []
    for step in range(200):
        with tf.GradientTape() as tape:
            loss = loss_fn(y, m(X, training=True))
        grads = tape.gradient(loss, m.trainable_variables)
        opt.apply_gradients(zip(grads, m.trainable_variables))
        losses.append(float(loss))
    curves[lr] = losses
    print(f"lr {lr:<6} final loss {losses[-1]:.4f}")

# ---- Plot loss-vs-step: the ball rolling into the valley ----
plt.figure(figsize=(6, 6))
for lr, losses in curves.items():
    plt.plot(losses, label=f"lr = {lr}")
plt.xlabel("step"); plt.ylabel("loss")
plt.title("loss rolling downhill   (w := w - lr*grad)")
plt.legend(); plt.tight_layout()
plt.savefig("loss_curve.png", dpi=120)

## TF06 · Did It Really Learn? Overfitting, Validation, and Regularization

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# --- Data: small, noisy 3-class blobs (small => overfits fast) ---
rng = np.random.default_rng(42)
centers = np.array([[0, 0], [4, 4], [8, 0]], dtype="float32")
y = rng.integers(0, 3, size=600)
X = centers[y] + rng.normal(0, 2.2, size=(600, 2)).astype("float32")

# Three-way split: train / val / test (no overlap)
Xtr, Xval, Xte = X[:360], X[360:480], X[480:]
ytr, yval, yte = y[:360], y[360:480], y[480:]

# --- Overfit on purpose: oversized net, NO regularization ---
overfit = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(256, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(3, activation="softmax"),
])
overfit.compile(optimizer="adam",
                loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h0 = overfit.fit(Xtr, ytr, validation_data=(Xval, yval), epochs=200, verbose=0)

# --- Diagnosis: train loss vs val loss (the overfitting signature) ---
def loss_curve(ax, hist, title):
    ax.plot(hist["loss"], label="train", color="#74b6ff")
    ax.plot(hist["val_loss"], label="val", color="#ff5fd2")
    ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()

# --- Regularized model: L2 + BatchNorm + Dropout ---
reg = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(256, activation="relu",
                 kernel_regularizer=keras.regularizers.l2(1e-3)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(3, activation="softmax"),
])
reg.compile(optimizer="adam",
            loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# --- Callbacks: stop early, keep the best weights, save to disk ---
cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                     restore_best_weights=True, verbose=1),
       keras.callbacks.ModelCheckpoint("best.keras", monitor="val_loss",
                                       save_best_only=True)]
h1 = reg.fit(Xtr, ytr, validation_data=(Xval, yval), epochs=200,
            callbacks=cbs, verbose=0)

## TF07 · Teaching a Machine to See: Convolutions and Cats vs Dogs

In [ ]:
# 07_cnn_catsdogs.py — Cats vs Dogs from raw photo folders with a CNN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

IMG_SIZE = (180, 180)
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

# --- Load images straight from folders (subfolder name = class) ---
train_ds = keras.utils.image_dataset_from_directory(
    "data/cats_dogs/train",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode="binary",
)
val_ds = keras.utils.image_dataset_from_directory(
    "data/cats_dogs/val",
    image_size=IMG_SIZE,
    batch_size=BATCH,
    label_mode="binary",
)
class_names = train_ds.class_names          # ['cats', 'dogs']

train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

# --- Augmentation: applied only during training ---
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augmentation")

# --- The CNN: 3 conv blocks -> flatten -> dense -> sigmoid ---
model = keras.Sequential([
    keras.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

history = model.fit(train_ds, validation_data=val_ds, epochs=15)

# --- Plot training vs validation accuracy ---
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend()
plt.title("Cats vs Dogs — CNN accuracy"); plt.savefig("cnn_acc.png")

# --- Predict on one held-out photo ---
PRETTY = {"cats": "cat", "dogs": "dog"}
img = keras.utils.load_img("data/test/cat.221.jpg", target_size=IMG_SIZE)
arr = keras.utils.img_to_array(img)
arr = tf.expand_dims(arr, 0)                # add batch dimension
prob = float(model.predict(arr)[0][0])
label = class_names[1] if prob > 0.5 else class_names[0]
conf = prob if prob > 0.5 else 1 - prob
print(f"predict cat.221.jpg -> {conf:.2f} {PRETTY[label]}")

## TF08 · Standing on Giants: Transfer Learning with MobileNetV2

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib.pyplot as plt

IMG_SIZE = (160, 160)
train_ds = keras.utils.image_dataset_from_directory(
    "data/cats_vs_dogs/train", image_size=IMG_SIZE, batch_size=32)
val_ds = keras.utils.image_dataset_from_directory(
    "data/cats_vs_dogs/val", image_size=IMG_SIZE, batch_size=32)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (preprocess_input(x), y)).cache().prefetch(AUTOTUNE)
val_ds   = val_ds.map(lambda x, y: (preprocess_input(x), y)).cache().prefetch(AUTOTUNE)

base = MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=keras.losses.BinaryCrossentropy(),
              metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])
hist = model.fit(train_ds, validation_data=val_ds, epochs=5)

base.trainable = True
fine_tune_at = len(base.layers) - 30
for layer in base.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss=keras.losses.BinaryCrossentropy(),
              metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])
hist_ft = model.fit(train_ds, validation_data=val_ds,
                    epochs=10, initial_epoch=hist.epoch[-1] + 1)

scratch_acc  = 0.910
frozen_acc   = max(hist.history["val_accuracy"])
finetune_acc = max(hist_ft.history["val_accuracy"])
plt.figure(figsize=(5, 5))
plt.bar(["from-scratch", "frozen head", "fine-tuned"],
        [scratch_acc, frozen_acc, finetune_acc],
        color=["#74b6ff", "#a878ff", "#ffd483"])
plt.ylim(0.85, 1.0)
plt.ylabel("val accuracy"); plt.title("Transfer learning")
plt.savefig("transfer_results.png", dpi=150)
model.save("cats_dogs_transfer.keras")

## TF09 · Where, Not Just What: Object Detection and Segmentation

In [ ]:
# 09_detection_segmentation.py — Predict stage: structured output (boxes + masks)
import numpy as np
import keras
import keras_hub
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# --- Load one street image, resize to the detector's input size (800x800) ---
img = keras.utils.load_img("street.jpg", target_size=(800, 800))
arr = keras.utils.img_to_array(img)              # (800, 800, 3), float32
batch = np.expand_dims(arr, axis=0)              # (1, 800, 800, 3)

# --- Pretrained detector: RetinaNet (ResNet-50 FPN) trained on COCO ---
detector = keras_hub.models.ImageObjectDetector.from_preset(
    "retinanet_resnet50_fpn_coco"
)

# --- Predict: dict of padded boxes / confidence / labels ---
pred = detector.predict(batch)
boxes  = pred["boxes"][0]            # (N, 4) -> [y1, x1, y2, x2] (yxyx)
scores = pred["confidence"][0]      # (N,)
labels = pred["labels"][0]          # (N,) integer class ids

# --- Keep confident detections (NMS already applied inside the model) ---
keep = scores > 0.5
boxes, scores, labels = boxes[keep], scores[keep], labels[keep]

COCO = {0: "person", 2: "car", 16: "dog", 9: "traffic light"}  # display map
for b, s, l in zip(boxes, scores, labels):
    print(f"detections: {COCO.get(int(l), int(l))} {s:.2f}")

# --- Draw boxes on the photo (left figure panel) ---
fig, (axL, axR) = plt.subplots(1, 2, figsize=(10, 5))
axL.imshow(img); axL.set_title("detection"); axL.axis("off")
for b, l in zip(boxes, labels):
    y1, x1, y2, x2 = b                       # boxes are yxyx
    color = "#ffd483" if int(l) == 0 else "#ff5fd2"
    axL.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                            fill=False, edgecolor=color, linewidth=2))
    axL.text(x1, y1 - 6, COCO.get(int(l), int(l)), color=color, fontsize=9)

# --- A U-Net for semantic masks (functional API) ---
def unet(num_classes=4):
    inp = keras.Input(shape=(128, 128, 3))
    c1 = keras.layers.Conv2D(32, 3, activation="relu", padding="same")(inp)
    p1 = keras.layers.MaxPool2D()(c1)
    c2 = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(p1)
    p2 = keras.layers.MaxPool2D()(c2)
    bott = keras.layers.Conv2D(128, 3, activation="relu", padding="same")(p2)
    u1 = keras.layers.concatenate([keras.layers.UpSampling2D()(bott), c2])  # skip
    u2 = keras.layers.concatenate([keras.layers.UpSampling2D()(u1), c1])    # skip
    out = keras.layers.Conv2D(num_classes, 1, activation="softmax")(u2)
    return keras.Model(inp, out)
model = unet(); batch_128 = np.expand_dims(keras.utils.img_to_array(keras.utils.load_img("street.jpg", target_size=(128, 128))), 0); mask = np.argmax(model.predict(batch_128), axis=-1)[0]

## TF10 · Reading the Pixels: OCR with CNN + RNN + CTC

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- 1. Load image paths and labels from filenames ---
DATA_DIR = "captchas"               # files like  b8x2k.png
IMG_W, IMG_H = 160, 50
BATCH = 16

paths = sorted(tf.io.gfile.glob(os.path.join(DATA_DIR, "*.png")))
labels = [os.path.splitext(os.path.basename(p))[0] for p in paths]
MAX_LEN = max(len(s) for s in labels)

# --- 2. Char vocabulary (index 0 reserved as CTC blank) ---
charset = sorted(set("".join(labels)))
char_to_num = layers.StringLookup(vocabulary=charset)   # 0 = [UNK]/blank
num_to_char = layers.StringLookup(
    vocabulary=char_to_num.get_vocabulary(), invert=True
)
N_CLASSES = char_to_num.vocabulary_size()   # includes reserved index 0

print(f"samples: {len(paths)}  charset: {''.join(charset)}")
print(f"max label len: {MAX_LEN}  classes (incl blank): {N_CLASSES}")

# --- 3. tf.data pipeline: decode, normalize, width-first ---
def encode(path, label):
    img = tf.io.decode_png(tf.io.read_file(path), channels=1)
    img = tf.image.resize(img, [IMG_H, IMG_W]) / 255.0
    img = tf.transpose(img, [1, 0, 2])          # (W, H, 1): width = time
    chars = tf.strings.unicode_split(label, "UTF-8")
    y = char_to_num(chars)                       # ints in [1, N_CLASSES)
    y = tf.pad(y, [[0, MAX_LEN - tf.shape(y)[0]]])   # pad with 0 (blank)
    return {"image": img, "label": y}

ds = tf.data.Dataset.from_tensor_slices((paths, labels))
ds = ds.map(encode, num_parallel_calls=tf.data.AUTOTUNE)
ds = ds.shuffle(512).batch(BATCH).prefetch(tf.data.AUTOTUNE)

# --- 4. CTC loss as a layer (logits in, add_loss, mask_index=0) ---
class CTCLayer(layers.Layer):
    def call(self, y_true, y_pred):
        n = tf.shape(y_pred)[0]
        out_len = tf.fill((n,), tf.shape(y_pred)[1])        # time steps
        tgt_len = tf.math.count_nonzero(y_true, axis=1, dtype="int32")
        loss = keras.ops.ctc_loss(y_true, y_pred, tgt_len, out_len)
        self.add_loss(tf.reduce_mean(loss))
        return y_pred

img_in = layers.Input(shape=(IMG_W, IMG_H, 1), name="image")
lab_in = layers.Input(shape=(MAX_LEN,), name="label")

# --- 5. CNN + RNN model: conv down, reshape, bi-LSTM, logits ---
x = layers.Conv2D(32, 3, padding="same", activation="relu")(img_in)
x = layers.MaxPooling2D(2)(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D(2)(x)                    # -> (W/4, H/4, 64)
new_w = IMG_W // 4
x = layers.Reshape((new_w, (IMG_H // 4) * 64))(x)   # width-wise sequence
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
y_pred = layers.Dense(N_CLASSES, name="char_logits")(x)   # logits, no softmax
out = CTCLayer()(lab_in, y_pred)

model = keras.Model([img_in, lab_in], out)
model.compile(optimizer=keras.optimizers.Adam(1e-3))   # loss added inside
model.fit(ds, epochs=20)

# --- 6. Greedy CTC decode for inference ---
pred_model = keras.Model(img_in, y_pred)
def decode(batch):
    logits = pred_model.predict(batch, verbose=0)
    seq_len = np.full(logits.shape[0], logits.shape[1])
    ids, _ = keras.ops.ctc_decode(logits, seq_len, strategy="greedy")
    ids = ids[0]                                  # (batch, max_len); blank=-1
    ids = tf.where(ids < 0, tf.zeros_like(ids), ids)
    texts = tf.strings.reduce_join(num_to_char(ids), axis=1)
    return [t.decode("utf-8").strip() for t in texts.numpy()]

## TF11 · From Words to Numbers: TextVectorization and Embeddings

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- A tiny labeled movie-review corpus (1 = positive, 0 = negative) ---
reviews = [
    "this movie was great", "a wonderful and excellent film",
    "fantastic acting truly brilliant", "i loved this great movie",
    "an excellent wonderful story", "brilliant and fantastic film",
    "this movie was terrible", "a boring and awful film",
    "horrible acting truly dull", "i hated this terrible movie",
    "an awful boring story", "dull and horrible film",
]
labels = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], dtype="float32")

# --- Text -> integers: build a vocabulary and pad to fixed length ---
vectorize = layers.TextVectorization(
    max_tokens=100,
    output_sequence_length=6,
)
vectorize.adapt(reviews)

vocab = vectorize.get_vocabulary()
print("vocab size:", len(vocab), "| first 8:", vocab[:8])

# --- Show one sentence mapped to integers ---
sample = "this movie was great"
ints = vectorize([sample]).numpy()[0]
print(f"'{sample}' -> {list(ints)}")

# --- Tiny model: learn 16-d embeddings via a sentiment signal ---
model = keras.Sequential([
    vectorize,
    layers.Embedding(input_dim=len(vocab), output_dim=16, name="embed"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(tf.constant(reviews), labels, epochs=40, verbose=0)

# --- Pull the learned word vectors out of the Embedding layer ---
emb = model.get_layer("embed").get_weights()[0]   # (vocab, 16)


def nearest(word, k=3):
    i = vocab.index(word)
    v = emb[i]
    sims = emb @ v / (np.linalg.norm(emb, axis=1) * np.linalg.norm(v) + 1e-9)
    order = [j for j in np.argsort(-sims) if j != i and vocab[j] not in ("", "[UNK]")]
    return [(vocab[j], round(float(sims[j]), 2)) for j in order[:k]]

print("nearest to great:", nearest("great"))

# --- 2D PCA map of the vocabulary (white figure) ---
from sklearn.decomposition import PCA
pos = {"great","wonderful","excellent","fantastic","brilliant","loved"}
words = [w for w in vocab if w not in ("", "[UNK]")]
pts = PCA(n_components=2).fit_transform(np.array([emb[vocab.index(w)] for w in words]))
print("PCA done — plotting", len(words), "words")

## TF12 · Positive or Negative? IMDb Sentiment with Embeddings

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

MAX_TOKENS, SEQ_LEN, BATCH = 10000, 250, 32

# 1 · Load IMDb reviews from folders of .txt files
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=BATCH,
    validation_split=0.2, subset="training", seed=42)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=BATCH,
    validation_split=0.2, subset="validation", seed=42)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=BATCH)

# 2 · TextVectorization: adapt on the training text only
vectorize = layers.TextVectorization(
    max_tokens=MAX_TOKENS, output_mode="int",
    output_sequence_length=SEQ_LEN)
train_text = train_ds.map(lambda x, y: x)
vectorize.adapt(train_text)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)

# 3 · Build the model: vectorize → embed → average → classify
model = keras.Sequential([
    keras.Input(shape=(), dtype=tf.string),
    vectorize,
    layers.Embedding(MAX_TOKENS, 16),
    layers.GlobalAveragePooling1D(),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])

# 4 · Compile: adam + binary crossentropy
model.compile(optimizer="adam",
              loss="binary_crossentropy", metrics=["accuracy"])

# 5 · Train, then evaluate on the held-out test split
history = model.fit(train_ds, validation_data=val_ds, epochs=10)
loss, acc = model.evaluate(test_ds)
print(f"test accuracy {acc:.3f}")

# 6 · Predict on two hand-typed reviews
reviews = ["a dull, lifeless mess", "an absolute triumph"]
probs = model.predict(tf.constant(reviews))
for r, p in zip(reviews, probs):
    print(f"'{r}' -> {p[0]:.2f} {'POSITIVE' if p[0] > 0.5 else 'NEGATIVE'}")

## TF13 · Reading in Order: RNN, LSTM, and GRU

In [ ]:
# 13_rnn_lstm.py — recurrent IMDb sentiment: LSTM / GRU / SimpleRNN
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers

# ---- Data: raw-text IMDb (same vectorizer as TF12) ----
train, test = tfds.load("imdb_reviews", split=["train", "test"],
                        as_supervised=True, batch_size=-1)
x_train_txt, y_train = tfds.as_numpy(train)
x_test_txt,  y_test  = tfds.as_numpy(test)
VOCAB, SEQ_LEN, EMB = 10000, 200, 64

vectorize = layers.TextVectorization(
    max_tokens=VOCAB, output_mode="int", output_sequence_length=SEQ_LEN)
vectorize.adapt(x_train_txt)        # adapt on raw training text
x_train, x_test = vectorize(x_train_txt), vectorize(x_test_txt)

# ---- Model factory: swap the recurrent layer to compare ----
def build_model(recurrent="lstm"):
    inp = keras.Input(shape=(SEQ_LEN,), dtype="int64")
    x = layers.Embedding(VOCAB, EMB, mask_zero=True)(inp)
    if recurrent == "simplernn":
        x = layers.Bidirectional(layers.SimpleRNN(64))(x)
    elif recurrent == "gru":
        x = layers.Bidirectional(layers.GRU(64, return_sequences=True))(x)
        x = layers.Bidirectional(layers.GRU(32))(x)
    else:  # lstm (default)
        x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
        x = layers.Bidirectional(layers.LSTM(32))(x)
    x = layers.Dense(64, activation="relu")(x)

# (head + return continue below)
    out = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inp, out)

# ---- Train + evaluate one configuration ----
def run(name, recurrent):
    model = build_model(recurrent)
    model.compile(optimizer="adam",
                  loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(x_train, y_train, epochs=5, batch_size=64,
              validation_split=0.2, verbose=2)
    _, acc = model.evaluate(x_test, y_test, verbose=0)
    print(f"{name:10s} test acc = {acc:.3f}")
    return model, acc

lstm_model, _ = run("LSTM", "lstm")
run("GRU", "gru")
run("SimpleRNN", "simplernn")

# ---- Word order, finally respected ----
for s in ["not good at all", "good, not bad at all"]:
    p = float(lstm_model.predict(vectorize([s]))[0][0])
    print(f'"{s}" -> {p:.2f} {"POS" if p>0.5 else "NEG"}')

## TF14 · Attention from Scratch: Query, Key, Value

In [ ]:
# 14_attention_scratch.py
# Scaled dot-product self-attention, by hand, in pure tf.matmul.
import numpy as np
import tensorflow as tf
from tensorflow import keras

# --- A tiny 5-word toy sentence -------------------------------
WORDS = ["the", "cat", "sat", "because", "it"]
rng = np.random.default_rng(14)
# Fake 16-dim embeddings: (batch=1, tokens=5, features=16)
x = rng.standard_normal((1, 5, 16)).astype("float32")
x = tf.constant(x)

# --- Scaled dot-product self-attention as a Keras Layer -------
class SelfAttention(keras.layers.Layer):
    def __init__(self, d_model=8, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model

    def build(self, input_shape):
        init_q = keras.initializers.GlorotUniform(seed=14)
        init_k = keras.initializers.GlorotUniform(seed=15)
        init_v = keras.initializers.GlorotUniform(seed=16)
        self.wq = keras.layers.Dense(self.d_model, use_bias=False,
                                     kernel_initializer=init_q)
        self.wk = keras.layers.Dense(self.d_model, use_bias=False,
                                     kernel_initializer=init_k)
        self.wv = keras.layers.Dense(self.d_model, use_bias=False,
                                     kernel_initializer=init_v)

    def call(self, x):
        q = self.wq(x)                                   # (1, 5, d)
        k = self.wk(x)                                   # (1, 5, d)
        v = self.wv(x)                                   # (1, 5, d)
        scores = tf.matmul(q, k, transpose_b=True)       # (1, 5, 5)
        scores /= tf.sqrt(tf.cast(self.d_model, tf.float32))
        weights = tf.nn.softmax(scores, axis=-1)         # rows sum to 1
        out = tf.matmul(weights, v)                      # (1, 5, d)
        return out, weights

# --- Run it once on the toy sentence -------------------------
layer = SelfAttention(d_model=8)
out, weights = layer(x)
attn = tf.squeeze(weights, axis=0).numpy()               # (5, 5)

print("attention matrix  (row = query word):\n")
header = "         " + "  ".join(f"{w:>7}" for w in WORDS)
print(header)
for i, w in enumerate(WORDS):
    row = "  ".join(f"{p:7.2f}" for p in attn[i])
    print(f"{w:>7}  {row}")

print(f"\nout shape: {tuple(out.shape)}   weights row sums:",
      np.round(attn.sum(axis=1), 3))

## TF15 · The Transformer Block: Multi-Head Attention, Norm, and Residuals

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


class PositionalEmbedding(layers.Layer):
    def __init__(self, seq_len, vocab_size, d_model, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = layers.Embedding(vocab_size, d_model)
        self.pos_emb = layers.Embedding(seq_len, d_model)
        self.seq_len = seq_len

    def call(self, x):
        positions = tf.range(start=0, limit=self.seq_len, delta=1)
        return self.token_emb(x) + self.pos_emb(positions)


class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(d_model),
        ])
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(rate)
        self.drop2 = layers.Dropout(rate)

    def call(self, x, training=False):
        attn, scores = self.mha(
            x, x, return_attention_scores=True, training=training)
        x = self.norm1(x + self.drop1(attn, training=training))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.drop2(ffn_out, training=training))
        return x, scores


VOCAB, SEQ_LEN, D_MODEL = 20000, 200, 128
NUM_HEADS, FF_DIM, N_BLOCKS = 8, 512, 1

inputs = keras.Input(shape=(SEQ_LEN,), dtype="int32")
x = PositionalEmbedding(SEQ_LEN, VOCAB, D_MODEL)(inputs)
block = TransformerBlock(D_MODEL, NUM_HEADS, FF_DIM)
x, attn_scores = block(x)

model = keras.Model(inputs, [x, attn_scores])
model.summary()

batch = tf.random.uniform((32, SEQ_LEN), maxval=VOCAB, dtype=tf.int32)
out, scores = model(batch, training=False)
print("block output shape", tuple(out.shape))
print("attention scores shape", tuple(scores.shape))
print("head 3 focuses on punctuation, head 7 on the subject")

## TF16 · A Transformer That Reads Reviews: Beating the LSTM

In [ ]:
# 16_transformer_classifier.py
# A Transformer encoder for IMDb sentiment — beating the LSTM.
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- config ---
VOCAB_SIZE = 20000
MAXLEN     = 200
EMBED_DIM  = 32
NUM_HEADS  = 2
FF_DIM     = 32

# --- data: IMDb as integer sequences, padded to a fixed length ---
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE)
x_train = keras.utils.pad_sequences(x_train, maxlen=MAXLEN)
x_test  = keras.utils.pad_sequences(x_test,  maxlen=MAXLEN)
# --- reused from TF15: token + position embedding ---
class PositionalEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.tok = layers.Embedding(vocab_size, embed_dim)
        self.pos = layers.Embedding(maxlen, embed_dim)
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1])

# 16_transformer_classifier.py
# A Transformer encoder for IMDb sentiment — beating the LSTM.
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- config ---
VOCAB_SIZE = 20000
MAXLEN     = 200
EMBED_DIM  = 32
NUM_HEADS  = 2
FF_DIM     = 32
# --- data: IMDb as integer sequences, padded to a fixed length ---
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(
    num_words=VOCAB_SIZE)
x_train = keras.utils.pad_sequences(x_train, maxlen=MAXLEN)
x_test  = keras.utils.pad_sequences(x_test,  maxlen=MAXLEN)
# --- reused from TF15: token + position embedding ---
class PositionalEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.tok = layers.Embedding(vocab_size, embed_dim)
        self.pos = layers.Embedding(maxlen, embed_dim)
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1])
        return self.tok(x) + self.pos(positions)

# --- reused from TF15: the Transformer encoder block ---
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.att  = layers.MultiHeadAttention(num_heads, embed_dim)
        self.ffn  = keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"),
             layers.Dense(embed_dim)])
        self.n1   = layers.LayerNormalization()
        self.n2   = layers.LayerNormalization()
        self.drop = layers.Dropout(0.1)
    def call(self, x, training=False):
        a = self.att(x, x)
        x = self.n1(x + self.drop(a, training=training))
        f = self.ffn(x)
        return self.n2(x + self.drop(f, training=training))

# --- build: Input -> PosEmb -> Block -> Pool -> Dropout -> Dense ---
inp = keras.Input(shape=(MAXLEN,))
h = PositionalEmbedding(MAXLEN, VOCAB_SIZE, EMBED_DIM)(inp)
h = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)(h)
h = layers.GlobalAveragePooling1D()(h)
h = layers.Dropout(0.3)(h)
out = layers.Dense(1, activation="sigmoid")(h)
model = keras.Model(inp, out)

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

# --- train: 3 epochs, validate on the test set ---
model.fit(x_train, y_train,
          batch_size=32, epochs=3,
          validation_data=(x_test, y_test))

# --- evaluate and compare to earlier baselines ---
_, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Transformer {acc:.3f}  vs  LSTM 0.886  vs  bag 0.872")

# --- predict on tricky negation / sarcasm, encoded with the IMDb word index ---
word_index = keras.datasets.imdb.get_word_index()
def encode(text):
    ids = [1] + [word_index.get(w, 2) + 3 for w in text.lower().split()]
    return [i if i < VOCAB_SIZE else 2 for i in ids]
tricky = ["not the worst film i've seen",
          "i expected to hate it but i loved every minute"]
xt = keras.utils.pad_sequences([encode(t) for t in tricky], maxlen=MAXLEN)
probs = model.predict(xt)
print("tricky →", [round(float(p), 2) for p in probs])

## TF17 · Teaching a Model to Write: Text Generation with Temperature

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# ---- 1. Load the Shakespeare corpus ----
path = keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt")
text = open(path, "r", encoding="utf-8").read()
vocab = sorted(set(text))
print(f"corpus: {len(text)} chars   vocab: {len(vocab)} symbols")

# ---- 2. Char <-> id lookup tables ----
char2id = {c: i for i, c in enumerate(vocab)}
id2char = np.array(vocab)
data = np.array([char2id[c] for c in text], dtype=np.int32)

# ---- 3. Window into (input, target) pairs ----
SEQ_LEN = 100
ds = tf.data.Dataset.from_tensor_slices(data)
ds = ds.batch(SEQ_LEN + 1, drop_remainder=True)
ds = ds.map(lambda chunk: (chunk[:-1], chunk[1:]))
ds = ds.shuffle(10000).batch(64, drop_remainder=True)
ds = ds.prefetch(tf.data.AUTOTUNE)

# ---- 4. A causal Transformer block ----
class TransformerBlock(layers.Layer):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.att = layers.MultiHeadAttention(n_heads, d_model // n_heads)
        self.ffn = keras.Sequential(
            [layers.Dense(d_ff, activation="relu"), layers.Dense(d_model)])
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x):
        a = self.att(x, x, use_causal_mask=True)   # look-ahead mask
        x = self.ln1(x + a)
        f = self.ffn(x)
        return self.ln2(x + f)

# ---- 5. Build the model (functional API) ----
V, D = len(vocab), 256
tok = layers.Input(shape=(None,), dtype="int32")
h   = layers.Embedding(V, D)(tok)
pos = layers.Embedding(SEQ_LEN, D)(layers.Lambda(
    lambda t: tf.range(tf.shape(t)[1]))(tok))
h   = TransformerBlock(D, 4, 512)(h + pos)
out = layers.Dense(V)(h)                      # logits over vocab

model = keras.Model(tok, out)
model.compile(optimizer="adam",
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))

# ---- 6. Train ----
model.fit(ds, epochs=30)

# ---- 7. Generate with temperature ----
def generate(seed, n=80, temp=1.0):
    ids = [char2id[c] for c in seed]
    for _ in range(n):
        x = tf.constant([ids[-SEQ_LEN:]])
        logits = model(x)[0, -1] / temp        # temperature scaling
        nxt = tf.random.categorical(logits[None], 1)[0, 0].numpy()
        ids.append(int(nxt))
    return "".join(id2char[i] for i in ids)

# ---- 8. Sample at three temperatures ----
for t in (0.2, 0.8, 1.2):
    print(f"\n--- temp {t} ---")
    print(generate("ROMEO: ", n=80, temp=t))

## TF18 · Machine Translation: Seq2Seq to a Transformer

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np, random, re, string

# --- Data: English -> Spanish pairs (target wrapped in [start]/[end]) ---
text_pairs = load_eng_spa_pairs()          # [("I love AI", "[start] me encanta la IA [end]"), ...]
random.shuffle(text_pairs)
train_pairs = text_pairs[:9000]
val_pairs   = text_pairs[9000:10000]

VOCAB, SEQ_LEN, D_MODEL = 15000, 20, 256

# --- Custom standardizer: lowercase but KEEP the [start]/[end] markers ---
strip = re.escape(string.punctuation.replace("[", "").replace("]", ""))
def spa_standardize(text):
    text = tf.strings.lower(text)
    return tf.strings.regex_replace(text, f"[{strip}]", "")

# --- Two vectorizers: English default, Spanish keeps the markers ---
eng_vec = layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=SEQ_LEN)
spa_vec = layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=SEQ_LEN + 1,
                                   standardize=spa_standardize)
eng_vec.adapt([e for e, s in train_pairs])
spa_vec.adapt([s for e, s in train_pairs])

# --- Positional embedding: token meaning + word order ---
class PositionalEmbedding(layers.Layer):
    def __init__(self, vocab, d_model, seq_len):
        super().__init__()
        self.tok = layers.Embedding(vocab, d_model, mask_zero=True)
        self.pos = layers.Embedding(seq_len, d_model)
    def call(self, x):
        pos = tf.range(tf.shape(x)[-1])
        return self.tok(x) + self.pos(pos)

HEADS = 8
KEY_DIM = D_MODEL // HEADS          # 32 per head — the idiomatic split

# --- Encoder block: self-attention (no mask) + FFN ---
class EncoderBlock(layers.Layer):
    def __init__(self, d_model, heads, key_dim, dff):
        super().__init__()
        self.attn = layers.MultiHeadAttention(heads, key_dim)
        self.ffn  = keras.Sequential([layers.Dense(dff, activation="relu"),
                                       layers.Dense(d_model)])
        self.n1, self.n2 = layers.LayerNormalization(), layers.LayerNormalization()
    def call(self, x):
        x = self.n1(x + self.attn(x, x))
        return self.n2(x + self.ffn(x))

# --- Decoder block: causal self-attn + cross-attn over encoder + FFN ---
class DecoderBlock(layers.Layer):
    def __init__(self, d_model, heads, key_dim, dff):
        super().__init__()
        self.self_attn  = layers.MultiHeadAttention(heads, key_dim)
        self.cross_attn = layers.MultiHeadAttention(heads, key_dim)
        self.ffn = keras.Sequential([layers.Dense(dff, activation="relu"),
                                      layers.Dense(d_model)])
        self.n1, self.n2, self.n3 = (layers.LayerNormalization() for _ in range(3))

    def call(self, x, enc):
        x = self.n1(x + self.self_attn(x, x, use_causal_mask=True))
        x = self.n2(x + self.cross_attn(query=x, key=enc, value=enc))
        return self.n3(x + self.ffn(x))

# --- Assemble the full encoder-decoder model (functional API) ---
enc_in = keras.Input(shape=(None,), dtype="int64", name="english")
dec_in = keras.Input(shape=(None,), dtype="int64", name="spanish")
enc_emb = PositionalEmbedding(VOCAB, D_MODEL, SEQ_LEN)(enc_in)
enc_out = EncoderBlock(D_MODEL, HEADS, KEY_DIM, 1024)(enc_emb)
dec_emb = PositionalEmbedding(VOCAB, D_MODEL, SEQ_LEN + 1)(dec_in)
dec_out = DecoderBlock(D_MODEL, HEADS, KEY_DIM, 1024)(dec_emb, enc_out)
logits  = layers.Dense(VOCAB)(layers.Dropout(0.2)(dec_out))
model = keras.Model([enc_in, dec_in], logits)

model.compile(optimizer="adam",
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=["accuracy"])
model.fit(make_ds(train_pairs), validation_data=make_ds(val_pairs), epochs=30)

# --- Greedy decode: predict one Spanish token at a time ---
spa_vocab = spa_vec.get_vocabulary()
def translate(sentence):
    src = eng_vec([sentence])
    out = "[start]"
    for i in range(SEQ_LEN):
        tgt = spa_vec([out])[:, :-1]
        preds = model([src, tgt])
        next_id = int(tf.argmax(preds[0, i]))
        word = spa_vocab[next_id]
        if word == "[end]": break
        out += " " + word
    return out.replace("[start]", "").strip()

# --- Evaluate with BLEU on held-out pairs ---
refs = [s.replace("[start]","").replace("[end]","").split() for _, s in val_pairs]
hyps = [translate(e).split() for e, _ in val_pairs]
bleu = corpus_bleu([[r] for r in refs], hyps)     # nltk corpus_bleu
print(f"BLEU {bleu:.2f}")
for e in ["I love machine learning", "where is the train station?"]:
    print(f"'{e}' -> '{translate(e)}'")

## TF19 · Fine-Tuning a Pretrained Transformer: BERT for Real Tasks

In [ ]:
# 19_bert_finetune.py
# Fine-tune a pretrained BERT classifier on a tiny IMDb subset.
import tensorflow as tf
from tensorflow import keras
import keras_hub
import tensorflow_datasets as tfds

# --- Load IMDb, but keep only 2,000 training examples on purpose ---
train_ds, test_ds = tfds.load(
    "imdb_reviews",
    split=["train", "test"],
    as_supervised=True,
)
train_ds = train_ds.take(2000).batch(16).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(16).prefetch(tf.data.AUTOTUNE)

# --- Load a pretrained BERT classifier (weights + tokenizer + head) ---
classifier = keras_hub.models.BertTextClassifier.from_preset(
    "bert_base_en_uncased",
    num_classes=2,
)

# --- Compile with a tiny learning rate (gentle steering) ---
classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

# --- Fine-tune for just 3 epochs on the raw-text dataset ---
classifier.fit(train_ds, epochs=3)

# --- Evaluate on the full untouched test set ---
loss, acc = classifier.evaluate(test_ds)
print(f"\nTest accuracy: {acc:.3f}")

# --- Predict on raw, unseen sentences ---
samples = [
    "a masterclass in tension, gripping from the first frame",
    "a dull, lifeless slog i could not wait to end",
]
# head returns logits -> softmax to probabilities, argmax for the label
logits = classifier.predict(samples)
probs = tf.nn.softmax(logits, axis=-1)
labels = tf.argmax(probs, axis=-1).numpy()
names = {0: "NEGATIVE", 1: "POSITIVE"}
for text, lab, p in zip(samples, labels, probs.numpy()):
    print(f'"{text[:32]}..." -> {names[lab]} ({p[lab]:.2f})')

# --- Save the whole fine-tuned model (BERT + preprocessor + head) ---
classifier.save("bert_imdb.keras")
print("Saved fine-tuned model to bert_imdb.keras")

## TF20 · Pulling Facts Out: Named Entity Recognition and Question Answering

In [ ]:
# 20_ner_qa.py — NER (token tags) + extractive QA (answer spans)
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ── BIO tag set: O = outside, B = begin entity, I = inside entity ──
TAGS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
tag2id = {t: i for i, t in enumerate(TAGS)}
id2tag = {i: t for t, i in tag2id.items()}
NUM_TAGS = len(TAGS)

# ── Toy CoNLL-style data: (tokens, tags) per sentence ──
SENTENCES = [
    (["Apple", "opened", "a", "store", "in", "Paris"],
     ["B-ORG", "O", "O", "O", "O", "B-LOC"]),
    (["Elon", "Musk", "leads", "Tesla"],
     ["B-PER", "I-PER", "O", "B-ORG"]),
    (["Sundar", "works", "at", "Google", "in", "London"],
     ["B-PER", "O", "O", "B-ORG", "O", "B-LOC"]),
]

# ── Word vocab: 0 = pad, 1 = unknown, real words start at 2 ──
VOCAB = sorted({w for toks, _ in SENTENCES for w in toks})
word2id = {w: i + 2 for i, w in enumerate(VOCAB)}
MAXLEN = 6

# ── Encode tokens + tags to padded integer arrays ──
def encode(tokens, tags):
    x = [word2id.get(w, 1) for w in tokens][:MAXLEN]   # 1 = <unk>
    y = [tag2id[t] for t in tags][:MAXLEN]
    x += [0] * (MAXLEN - len(x))                       # 0 = <pad>
    y += [0] * (MAXLEN - len(y))
    return x, y

X = np.array([encode(t, g)[0] for t, g in SENTENCES])
Y = np.array([encode(t, g)[1] for t, g in SENTENCES])

# ── Token-classification model: encoder → tag per position ──
model = keras.Sequential([
    layers.Input(shape=(MAXLEN,)),
    layers.Embedding(len(word2id) + 2, 32, mask_zero=True),
    layers.Bidirectional(layers.LSTM(32, return_sequences=True)),
    layers.TimeDistributed(layers.Dense(NUM_TAGS, activation="softmax")),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(X, Y, epochs=40, verbose=2)

# ── Decode: argmax tags per token, keep the non-O runs ──
def tag_sentence(tokens):
    x = np.array([encode(tokens, ["O"] * len(tokens))[0]])
    probs = model.predict(x, verbose=0)[0]
    ids = probs.argmax(-1)[:len(tokens)]
    return [(w, id2tag[i]) for w, i in zip(tokens, ids)]

for w, t in tag_sentence(["Apple", "opened", "a", "store", "in", "Paris"]):
    if t != "O":
        print(f"  {w:8s} -> {t}")

# ── Extractive QA: pretrained BERT + fine-tuned start/end head ──
import keras_hub
backbone = keras_hub.models.BertBackbone.from_preset("bert_base_en_uncased")
preproc  = keras_hub.models.BertTextClassifierPreprocessor.from_preset(
               "bert_base_en_uncased", sequence_length=64)
span_head = layers.Dense(2)                 # 2 logits per token: start, end
span_head.build((None, None, backbone.output_shape["sequence_output"][-1]))
span_head.load_weights("qa_span_head.weights.h5")   # fine-tuned earlier

def answer(question, context):
    x = preproc((question, context))        # → token_ids, segment_ids, padding_mask
    seq = backbone(x)["sequence_output"]    # (1, seq_len, hidden)

def answer(question, context):
    x = preproc((question, context))
    seq = backbone(x)["sequence_output"]
    logits = span_head(seq)[0]                       # (seq_len, 2)
    s = int(tf.argmax(logits[:, 0])); e = int(tf.argmax(logits[:, 1]))
    ids = x["token_ids"][0][min(s, e):max(s, e) + 1]
    return preproc.tokenizer.detokenize(ids).numpy().decode()

print(answer("Who founded Tesla?",
             "Tesla was founded by Elon Musk in 2003"))

## TF21 · Predicting the Future: Windowed LSTM for Time Series

In [ ]:
# 21_timeseries_lstm.py — Windowed LSTM for time-series forecasting (tf.keras)
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# --- Load: Jena weather CSV, one column, subsample to hourly ---
df = pd.read_csv("jena_weather.csv")
series = df["T (degC)"].astype("float32").values[::6]   # 10-min -> hourly, (N,)

# --- Split by TIME (never shuffle a time series) ---
N = len(series)
split = int(N * 0.70)
train, test = series[:split], series[split:]

# --- Normalize using TRAIN stats only ---
mu, sigma = train.mean(), train.std()
train_n = (train - mu) / sigma
test_n  = (test  - mu) / sigma

# --- Sliding windows: last 72 hours -> next hour ---
WINDOW = 72
def make_ds(data):
    return keras.utils.timeseries_dataset_from_array(
        data=data[:-1], targets=data[WINDOW:],
        sequence_length=WINDOW, batch_size=32, shuffle=False)
train_ds, test_ds = make_ds(train_n), make_ds(test_n)

# --- Build: LSTM(64) -> Dense(1), one-step predictor ---
model = keras.Sequential([
    keras.layers.Input((WINDOW, 1)),
    keras.layers.LSTM(64),
    keras.layers.Dense(1)])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

# --- Train ---
model.fit(train_ds, validation_data=test_ds, epochs=10)

# --- Multi-step forecast: autoregressive, 24 steps (a day) ahead ---
DELAY = 24
window = test_n[:WINDOW].reshape(1, WINDOW, 1)
preds = []
for _ in range(DELAY):
    nxt = model.predict(window, verbose=0)[0, 0]
    preds.append(nxt)
    window = np.roll(window, -1, axis=1)
    window[0, -1, 0] = nxt
preds = np.array(preds) * sigma + mu               # back to °C

# --- Evaluate day-ahead vs naive 24h persistence (real °C) ---
actual_h = test[WINDOW:WINDOW + DELAY]              # the 24 true hours
model_mae = np.mean(np.abs(preds - actual_h))
naive     = np.mean(np.abs(test[DELAY:] - test[:-DELAY]))  # t+24h = t
print(f"day-ahead MAE {model_mae:.2f} degC")
print(f"naive 24h baseline {naive:.2f} degC")

# --- Plot: actual (blue) vs forecast (gold dashed) ---
actual = test[WINDOW - 72:WINDOW + DELAY]           # 96 hours of context+horizon
plt.figure(figsize=(8, 4))
plt.plot(range(96), actual, color="#1f6feb", label="actual")
plt.plot(range(72, 96), preds, "--", color="#d9a441", label="forecast")
plt.axvspan(72, 96, color="#d9a441", alpha=0.12); plt.legend(); plt.xlabel("hour"); plt.ylabel("T (degC)"); plt.tight_layout(); plt.savefig("forecast.png")

## TF22 · What Should You Watch Next? Two-Tower Recommendations

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- 1. MovieLens-style interactions: who watched what ---
#     Three users with overlapping Pixar habits so the
#     Pixar titles co-occur and become neighbors.
user_ids  = np.array(["42","42","42","7","7","7","13","13","13"])
movie_ids = np.array(["Toy Story","Up","WALL-E",
                      "Up","Finding Nemo","Ratatouille",
                      "Toy Story","Finding Nemo","WALL-E"])
all_movies = np.array(["Toy Story","Up","WALL-E","Finding Nemo",
                       "Ratatouille","Heat","Casino","The Matrix"])

EMB_DIM, BATCH = 32, 3
ds = tf.data.Dataset.from_tensor_slices((user_ids, movie_ids))
ds = ds.shuffle(9).batch(BATCH, drop_remainder=True).repeat()

# --- 2. StringLookup vocabularies (ids -> integer indices) ---
user_lookup  = layers.StringLookup()
movie_lookup = layers.StringLookup()
user_lookup.adapt(user_ids)
movie_lookup.adapt(all_movies)

# --- 3. Tower factory: Embedding -> Dense -> 32-d vector ---
def make_tower(lookup, name):
    inp = keras.Input(shape=(), dtype=tf.string, name=name)
    idx = lookup(inp)
    emb = layers.Embedding(lookup.vocabulary_size(), EMB_DIM)(idx)
    out = layers.Dense(EMB_DIM)(emb)
    return keras.Model(inp, out, name=name + "_tower")

# --- 4. Build the two towers (separate weights, same space) ---
user_tower  = make_tower(user_lookup,  "user")
movie_tower = make_tower(movie_lookup, "movie")

# --- 5. Two-tower model: dot-product score + in-batch softmax ---
class TwoTower(keras.Model):
    def __init__(self, ut, mt):
        super().__init__()
        self.user_tower, self.movie_tower = ut, mt
    def train_step(self, data):
        users, movies = data
        with tf.GradientTape() as tape:
            u = self.user_tower(users)                   # (B, 32)
            m = self.movie_tower(movies)                 # (B, 32)
            scores = tf.matmul(u, m, transpose_b=True)   # (B, B)
            labels = tf.range(tf.shape(scores)[0])       # diagonal = real
            loss = keras.losses.sparse_categorical_crossentropy(
                labels, scores, from_logits=True)

        g = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(g, self.trainable_variables))
        return {"loss": tf.reduce_mean(loss)}

model = TwoTower(user_tower, movie_tower)
model.compile(optimizer=keras.optimizers.Adam(0.05))
model.fit(ds, steps_per_epoch=3, epochs=5, verbose=2)

# --- 6. Serve: embed all movies, recommend top-k for one user ---
movie_vecs = tf.math.l2_normalize(movie_tower(all_movies), axis=1)  # (N,32)
u42 = tf.math.l2_normalize(user_tower(np.array(["42"])), axis=1)
cos = tf.matmul(u42, movie_vecs, transpose_b=True)[0]               # cosine

seen  = {"Toy Story", "Up", "WALL-E"}        # User 42's history
order = tf.argsort(cos, direction="DESCENDING").numpy()
recs  = [(all_movies[i], float(cos[i])) for i in order
         if all_movies[i] not in seen][:3]
print("User 42 liked Toy Story, Up -> recommends:")
for name, s in recs:
    print(f"  {name:<14} {s:.2f}")

## TF23 · Learning by Doing: A Policy-Gradient Agent for CartPole

In [ ]:
import gymnasium as gym
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

# --- Environment + policy network ---
env = gym.make("CartPole-v1")
n_actions = env.action_space.n          # 2: push left or right

policy = keras.Sequential([
    layers.Input(shape=(4,)),           # state: pos, vel, angle, ang-vel
    layers.Dense(24, activation="relu"),
    layers.Dense(n_actions, activation="softmax"),
])
optimizer = keras.optimizers.Adam(learning_rate=1e-2)
gamma = 0.99                            # discount factor

# --- Discounted, standardized returns (whole helper on one page) ---
def discount_rewards(rewards):
    out, running = np.zeros_like(rewards, dtype=np.float32), 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running
        out[t] = running
    out = (out - out.mean()) / (out.std() + 1e-8)   # standardize
    return out

# --- Play one episode (called inside the tape later) ---
def run_episode():
    state, _ = env.reset()
    log_probs, rewards = [], []
    done = False
    while not done:
        s = tf.convert_to_tensor(state[None], dtype=tf.float32)
        probs = policy(s)                          # (1, 2) action probs
        action = tf.random.categorical(tf.math.log(probs), 1)[0, 0]
        log_probs.append(tf.math.log(probs[0, action]))
        state, reward, terminated, truncated, _ = env.step(int(action))
        rewards.append(reward)
        done = terminated or truncated
    return log_probs, rewards

# (REINFORCE loss helper continues on page 3, defined as its own
#  whole section there — nothing here straddles the page break)

# --- REINFORCE loss for one episode ---
def reinforce_loss(log_probs, rewards):
    returns = discount_rewards(rewards)
    lp = tf.stack(log_probs)
    returns = tf.convert_to_tensor(returns, dtype=tf.float32)
    return -tf.reduce_sum(lp * returns)            # push up high-return actions

# --- Train: drive each gradient step by hand ---
episode_rewards = []
for episode in range(1, 601):
    with tf.GradientTape() as tape:
        log_probs, rewards = run_episode()         # forward calls recorded
        loss = reinforce_loss(log_probs, rewards)
    grads = tape.gradient(loss, policy.trainable_variables)
    optimizer.apply_gradients(zip(grads, policy.trainable_variables))

    total = sum(rewards)
    episode_rewards.append(total)
    avg100 = np.mean(episode_rewards[-100:])
    if episode % 10 == 0:
        print(f"episode {episode:3d}  reward {int(total):3d}  avg100 {avg100:6.1f}")
    if avg100 >= 475.0 and episode >= 100:
        print(f"episode {episode}  reward {int(total)}  (solved!)")
        break

# --- Plot the learning curve ---
plt.figure(figsize=(6, 6)); plt.plot(episode_rewards, color="#a878ff")
plt.xlabel("episode"); plt.ylabel("episode reward"); plt.title("REINFORCE on CartPole-v1"); plt.tight_layout(); plt.savefig("cartpole_reward_curve.png", dpi=150)

## TF24 · Making Things Up: GANs, VAEs, and a Diffusion Glimpse

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

# ---- Data: MNIST scaled to [-1, 1] for tanh output ----
(x_train, y_train), (_, _) = keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") - 127.5) / 127.5
x_train = np.expand_dims(x_train, -1)          # (60000, 28, 28, 1)
BUFFER, BATCH, NOISE_DIM = 60000, 256, 100
dataset = (tf.data.Dataset.from_tensor_slices(x_train)
           .shuffle(BUFFER).batch(BATCH))

# ---- Generator: noise -> 28x28 image ----
def make_generator():
    m = keras.Sequential([
        layers.Input(shape=(NOISE_DIM,)),
        layers.Dense(7 * 7 * 256, use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(),
        layers.Reshape((7, 7, 256)),
        layers.Conv2DTranspose(128, 5, strides=1, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(),
        layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(),
        layers.Conv2DTranspose(1, 5, strides=2, padding="same",
                               use_bias=False, activation="tanh"),
    ], name="generator")
    return m


# ---- Discriminator: 28x28 image -> one logit ----
def make_discriminator():
    m = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(64, 5, strides=2, padding="same"),
        layers.LeakyReLU(), layers.Dropout(0.3),
        layers.Conv2D(128, 5, strides=2, padding="same"),
        layers.LeakyReLU(), layers.Dropout(0.3),
        layers.Flatten(), layers.Dense(1),
    ], name="discriminator")
    return m

generator = make_generator()
discriminator = make_discriminator()

# ---- Losses + two optimizers ----
bce = keras.losses.BinaryCrossentropy(from_logits=True)

def disc_loss(real_out, fake_out):
    real_l = bce(tf.ones_like(real_out), real_out)
    fake_l = bce(tf.zeros_like(fake_out), fake_out)
    return 0.5 * (real_l + fake_l)         # averaged -> equilibrium ~ ln2

def gen_loss(fake_out):
    return bce(tf.ones_like(fake_out), fake_out)   # wants fakes called "real"


gen_opt = keras.optimizers.Adam(1e-4)
disc_opt = keras.optimizers.Adam(1e-4)

# ---- Custom train step: two GradientTapes ----
@tf.function
def train_step(real_images):
    noise = tf.random.normal([tf.shape(real_images)[0], NOISE_DIM])
    with tf.GradientTape() as g_tape, tf.GradientTape() as d_tape:
        fakes = generator(noise, training=True)
        real_out = discriminator(real_images, training=True)
        fake_out = discriminator(fakes, training=True)
        g_l = gen_loss(fake_out)
        d_l = disc_loss(real_out, fake_out)
    g_grad = g_tape.gradient(g_l, generator.trainable_variables)
    d_grad = d_tape.gradient(d_l, discriminator.trainable_variables)
    gen_opt.apply_gradients(zip(g_grad, generator.trainable_variables))
    disc_opt.apply_gradients(zip(d_grad, discriminator.trainable_variables))
    return d_l, g_l


# ---- Training loop ----
EPOCHS = 50
seed = tf.random.normal([16, NOISE_DIM])        # fixed noise for the grid

for epoch in range(1, EPOCHS + 1):
    for batch in dataset:
        d_l, g_l = train_step(batch)
    if epoch % 10 == 0:
        print(f"epoch {epoch}: D_loss {d_l:.2f} G_loss {g_l:.2f}")


# ---- Sample a 4x4 grid from the trained generator ----
samples = generator(seed, training=False)
samples = (samples + 1.0) / 2.0                 # back to [0, 1]
fig, axes = plt.subplots(4, 4, figsize=(4, 4))
for img, ax in zip(samples, axes.flat):
    ax.imshow(tf.squeeze(img), cmap="gray")
    ax.axis("off")
plt.tight_layout(); plt.savefig("gan_samples.png")
print("16 digit samples saved")


# ---- A brief VAE ----
LATENT = 2
enc_in = keras.Input(shape=(28, 28, 1))
h = layers.Flatten()(enc_in)
h = layers.Dense(256, activation="relu")(h)
z_mean = layers.Dense(LATENT)(h)
z_logvar = layers.Dense(LATENT)(h)
encoder = keras.Model(enc_in, [z_mean, z_logvar], name="encoder")

def sample_z(z_mean, z_logvar):                  # reparameterization trick
    eps = tf.random.normal(tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_logvar) * eps

dec_in = keras.Input(shape=(LATENT,))
d = layers.Dense(256, activation="relu")(dec_in)
d = layers.Dense(28 * 28, activation="sigmoid")(d)
dec_out = layers.Reshape((28, 28, 1))(d)
decoder = keras.Model(dec_in, dec_out, name="decoder")


# ---- VAE custom train step (recon + KL) ----
vae_opt = keras.optimizers.Adam(1e-3)

@tf.function
def vae_step(x):
    with tf.GradientTape() as tape:
        z_mean, z_logvar = encoder(x)
        z = sample_z(z_mean, z_logvar)
        x_hat = decoder(z)
        recon = tf.reduce_sum(keras.losses.binary_crossentropy(x, x_hat), axis=(1, 2))
        kl = -0.5 * tf.reduce_sum(1 + z_logvar - tf.square(z_mean) - tf.exp(z_logvar), axis=1)
        loss = tf.reduce_mean(recon + kl)
    vars_ = encoder.trainable_variables + decoder.trainable_variables
    grads = tape.gradient(loss, vars_)
    vae_opt.apply_gradients(zip(grads, vars_))
    return loss


vae_ds = (tf.data.Dataset.from_tensor_slices((x_train + 1.0) / 2.0)
          .shuffle(BUFFER).batch(BATCH))
for epoch in range(1, 31):
    for batch in vae_ds:
        l = vae_step(batch)
    if epoch % 10 == 0:
        print(f"vae epoch {epoch}: loss {l:.1f}")

# ---- Latent morph: a 3 into an 8 ----
imgs01 = (x_train + 1.0) / 2.0
three = imgs01[np.where(y_train == 3)[0][0]][None]
eight = imgs01[np.where(y_train == 8)[0][0]][None]
z3, _ = encoder(three); z8, _ = encoder(eight)
fig, axes = plt.subplots(1, 10, figsize=(10, 1))
for i, t in enumerate(np.linspace(0, 1, 10)):
    z = z3 * (1 - t) + z8 * t                    # interpolate codes
    axes[i].imshow(tf.squeeze(decoder(z)), cmap="gray"); axes[i].axis("off")
plt.savefig("vae_morph.png")
print("latent morph saved: 3 -> 8")


# ---- Diffusion idea: forward noising, reverse denoising (concept) ----
x0 = imgs01[0]                                   # one clean digit
betas = np.linspace(1e-4, 0.2, 5)               # 5 noise steps
noised, x = [x0], x0
for b in betas:
    x = np.sqrt(1 - b) * x + np.sqrt(b) * np.random.randn(*x.shape)
    noised.append(np.clip(x, 0, 1))
# reverse = train a model to predict & remove the noise (next series)
print("diffusion: 5 forward noise steps illustrated")

## TF25 · Under the Hood: GradientTape, Autodiff, and Custom Training

In [ ]:
import time
import numpy as np
import tensorflow as tf
from tensorflow import keras

# --- 1 · Sanity check: d/dx x**3 should be 3x**2 ---
x = tf.Variable(2.0)
with tf.GradientTape() as tape:
    y = x ** 3
dy_dx = tape.gradient(y, x)
print(f"d/dx x**3 at x=2 -> {dy_dx.numpy()}  (analytic {3 * 2**2})")

# --- 2 · Nested tapes: second derivative d2/dx2 x**3 = 6x ---
with tf.GradientTape() as t2:
    with tf.GradientTape() as t1:
        y = x ** 3
    d1 = t1.gradient(y, x)
d2 = t2.gradient(d1, x)
print(f"1st={d1.numpy()} (3x^2)  2nd={d2.numpy()} (6x)")

# --- 3 · Physics least-squares with tf.linalg ---
t = tf.linspace(0.0, 5.0, 50)[:, None]
noise = tf.random.normal([50, 1], stddev=0.02, seed=0)
b = 3.14 * tf.cos(t) - 0.21 * t + noise          # k*cos(t) - damping*t
A = tf.concat([tf.cos(t), -t], axis=1)           # design matrix [cos(t), -t]

k, damping = tf.unstack(tf.linalg.lstsq(A, b)[:, 0])
resid = tf.reduce_mean((A @ tf.stack([[k], [damping]]) - b) ** 2)
print(f"fitted params k={k:.2f}, damping={damping:.2f}, residual={resid:.1e}")

# --- 4 · Custom train_step: subclass keras.Model (Keras 3 idiom) ---
class LinearModel(keras.Model):
    def __init__(self):
        super().__init__()
        self.dense = keras.layers.Dense(1)
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.mae_metric = keras.metrics.MeanAbsoluteError(name="mae")
    def call(self, x):
        return self.dense(x)
    def train_step(self, data):
        xb, yb = data
        with tf.GradientTape() as tape:
            pred = self(xb, training=True)
            loss = self.compute_loss(y=yb, y_pred=pred)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.loss_tracker.update_state(loss)
        self.mae_metric.update_state(yb, pred)
        return {"loss": self.loss_tracker.result(),
                "mae": self.mae_metric.result()}
    @property
    def metrics(self):
        return [self.loss_tracker, self.mae_metric]

# --- 5 · Wire it up like any Keras model ---
X = tf.random.normal([512, 4], seed=1)
W = tf.constant([[1.0], [-2.0], [0.5], [3.0]])
Y = X @ W + 0.1
model = LinearModel()
model.compile(optimizer=keras.optimizers.Adam(0.05), loss="mse")
model.fit(X, Y, epochs=8, batch_size=64, verbose=2)

# --- 6 · @tf.function graph speedup ---
def step(z):
    with tf.GradientTape() as tp:
        out = tf.reduce_sum(tf.square(model(z)))
    return tp.gradient(out, model.trainable_variables)
fast = tf.function(step)
times = {}
for fn, name in [(step, "eager"), (fast, "@tf.function")]:
    t0 = time.time(); [fn(X) for _ in range(1000)]
    times[name] = time.time() - t0; print(name, round(times[name], 3))
print(f"@tf.function: {times['eager'] / times['@tf.function']:.1f}x faster")

## TF26 · Shipping It: SavedModel, TF Serving, and TensorFlow.js

In [ ]:
# 26_serving.py — Save & Deploy: SavedModel, TF Serving, TF.js
import tensorflow as tf
from tensorflow import keras

# 1) Load the sentiment model trained in an earlier episode (20 features -> 1 prob)
model = keras.models.load_model("sentiment.keras")
print(model.input_shape, model.output_shape)

# 2) Export the language-neutral SavedModel into a VERSIONED folder.
#    The trailing "1" is the version TF Serving looks for.
model.export("saved_model/1")
print("exported ->", "saved_model/1")

# 3) Inspect the serving signature from the command line:
#    (run in a shell; shows the input/output contract callers rely on)
inspect = r"""
saved_model_cli show \
  --dir saved_model/1 \
  --tag_set serve \
  --signature_def serving_default
"""
print(inspect)
# Expected: inputs 'inputs' float32 [-1, 20]
#           outputs 'output_0' float32 [-1, 1]
# -> this exact shape is what every client must send.

# 4) Serve the SavedModel over REST with TF Serving (one command).
#    NOTE: model_base_path is the folder ABOVE the version "1".
serve = r"""
tensorflow_model_server \
  --rest_api_port=8501 \
  --model_name=sentiment \
  --model_base_path=$(pwd)/saved_model
"""
print(serve)

# 5) Call the live endpoint with a plain HTTP POST (curl):
curl_call = r"""
curl -X POST http://localhost:8501/v1/models/sentiment:predict \
  -H "Content-Type: application/json" \
  -d '{"instances": [[0.12,0.03,0.55,0.91,0.02,0.44,0.78,0.10,
                      0.33,0.67,0.05,0.88,0.21,0.49,0.73,0.14,
                      0.60,0.27,0.95,0.08]]}'
"""
print(curl_call)   # -> {"predictions": [[0.96]]}

# 6) Convert the SAME SavedModel to run in the browser with TF.js:
convert = r"""
tensorflowjs_converter \
  --input_format=tf_saved_model \
  saved_model/1 web_model/
"""

## TF27 · AI in Your Pocket: TensorFlow Lite and Quantization

In [ ]:
import os, time
import numpy as np
import tensorflow as tf
from tensorflow import keras

# ---- Load data (same preprocessing the model was trained with) ----
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = (x_train / 255.0).astype("float32")[..., None]
x_test  = (x_test  / 255.0).astype("float32")[..., None]

# ---- Load the already-trained Keras CNN (Deploy stage: no training here) ----
model = keras.models.load_model("mnist_cnn.keras")

# ---- 1) Plain float32 TFLite export (the honest baseline) ----
conv = tf.lite.TFLiteConverter.from_keras_model(model)
float_tflite = conv.convert()
with open("mnist_float.tflite", "wb") as f:
    f.write(float_tflite)
float_bytes = os.path.getsize("mnist_float.tflite")
print(f"wrote mnist_float.tflite   ({float_bytes:,} bytes)")

# ---- 2) Representative dataset: ~100 real samples to calibrate int8 ranges ----
def representative_dataset():
    for img in x_train[:100]:
        yield [img[None, ...].astype("float32")]
# (int8 conversion continues on page 2)

# ---- 2b) Full int8 quantization with the representative dataset ----
q = tf.lite.TFLiteConverter.from_keras_model(model)
q.optimizations = [tf.lite.Optimize.DEFAULT]
q.representative_dataset = representative_dataset
q.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
q.inference_input_type  = tf.int8
q.inference_output_type = tf.int8
int8_tflite = q.convert()
with open("mnist_int8.tflite", "wb") as f:
    f.write(int8_tflite)
int8_bytes = os.path.getsize("mnist_int8.tflite")
print(f"wrote mnist_int8.tflite    ({int8_bytes:,} bytes)")

# ---- 3) Run a .tflite file with the Interpreter (as a device would) ----
def run_tflite(path, images, tag):
    interp = tf.lite.Interpreter(model_path=path)
    interp.allocate_tensors()
    inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
    preds, t0 = [], time.time()
    for img in images:
        x = img[None, ...]
        if inp["dtype"] == np.int8:
            scale, zp = inp["quantization"]
            x = np.clip((x / scale + zp).round(), -128, 127).astype(np.int8)
        interp.set_tensor(inp["index"], x)
        interp.invoke()
        preds.append(int(np.argmax(interp.get_tensor(out["index"]))))
    ms = (time.time() - t0) / len(images) * 1000.0

    print(f"running {tag} interpreter on {len(images)} samples ... done  (avg {ms:4.1f} ms/img)")
    return np.array(preds), ms

# ---- 4) Compare size, accuracy, latency ----
fp, fms = run_tflite("mnist_float.tflite", x_test, "float")
ip, ims = run_tflite("mnist_int8.tflite",  x_test, "int8 ")
acc_f = (fp == y_test).mean()
acc_i = (ip == y_test).mean()
print(f"float model {float_bytes/1e6:.1f} MB -> int8 {int8_bytes/1e6:.1f} MB")
print(f"accuracy {acc_f:.3f} -> {acc_i:.3f} | latency {fms:.1f} ms -> {ims:.1f} ms")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 5))
labels = ["size (MB)", "latency (ms)"]
ax.bar([0, 1], [float_bytes/1e6, fms], width=0.35, label="float32", color="#c9c9c9")
ax.bar([0.4, 1.4], [int8_bytes/1e6, ims], width=0.35, label="int8", color="#ffd483")
ax.set_xticks([0.2, 1.2]); ax.set_xticklabels(labels); ax.legend()
plt.tight_layout(); plt.savefig("tflite_compare.png", dpi=130); print("saved figure tflite_compare.png")

## TF28 · Closing the Loop: A Full Project from Collect to Predict

In [ ]:
# 28_capstone.py — the full pipeline, one script: collect → predict
import tensorflow as tf
import tensorflow_text as tf_text     # needed for tflite text ops
from tensorflow import keras
import keras_hub
import numpy as np
import matplotlib.pyplot as plt

SEQ_LEN, BATCH, EPOCHS = 256, 32, 4
PRESET = "distil_bert_base_en_uncased"

# ── Stops 1+2 · Collect + Preprocess ─────────────────────
train_ds, val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=BATCH, validation_split=0.2,
    subset="both", seed=42,
)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
print(f"[1+2] data batched -> {BATCH}/batch, cached + prefetched")

# ── Stop 3 · Build a pretrained transformer, freeze it ───
classifier = keras_hub.models.TextClassifier.from_preset(
    PRESET, num_classes=2, activation=None, sequence_length=SEQ_LEN,
)
classifier.backbone.trainable = False   # freeze giant, train head

# ── Stop 4 · Compile + Train (the honest loop) ───────────
classifier.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(
        "best.keras", monitor="val_accuracy", save_best_only=True),
]
history = classifier.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
print(f"[4] trained -> best val_accuracy "
      f"{max(history.history['val_accuracy']):.2f}")

# ── Stop 5 · Evaluate (confusion matrix, not just acc) ───
y_true = np.concatenate([y for _, y in val_ds], axis=0)
logits = classifier.predict(val_ds)
y_pred = np.argmax(logits, axis=1)
cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=2).numpy()
print(f"[5] confusion matrix:\n{cm}")
plt.figure(figsize=(4, 4)); plt.imshow(cm, cmap="Purples")
plt.xlabel("predicted"); plt.ylabel("true")
plt.title("validation confusion matrix"); plt.savefig("cm.png")

# ── Stop 6 · Save (SavedModel — the TF Serving artifact) ─
classifier.export("saved_model/imdb")
print("[6] saved -> saved_model/imdb (SavedModel for servers)")

# ── Stop 7 · Deploy (TFLite — carry the text ops along) ──
converter = tf.lite.TFLiteConverter.from_saved_model("saved_model/imdb")
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,     # native tflite kernels
    tf.lite.OpsSet.SELECT_TF_OPS,       # fallback for tf.text ops
]
converter.allow_custom_ops = True
tflite_model = converter.convert()
with open("model.tflite", "wb") as f:
    f.write(tflite_model)
size_mb = len(tflite_model) / 1e6
print(f"[7] tflite {size_mb:.1f} MB -> model.tflite (for the edge)")

# ── Stop 8 · Predict (one fresh raw sentence, on-device) ─
sample = tf.constant(["An absolute triumph — I was moved to tears."])
interp = tf_text.InterpreterWithCustomOps(
    model_path="model.tflite",
    custom_op_registerers=tf_text.tflite_registrar.SELECT_TFTEXT_OPS,
)
interp.allocate_tensors()
runner = interp.get_signature_runner("serving_default")
out = runner(**{list(interp.get_input_details()[0].keys())[0]: sample})
logit = list(out.values())[0][0]
prob = tf.nn.softmax(logit).numpy()
label = "POSITIVE" if np.argmax(prob) == 1 else "NEGATIVE"
print(f"[8] prediction: {label} {prob.max():.2f}")

# ── The loop is closed: collect → preprocess → build →
#    train → evaluate → save → deploy → predict ──────────
print("pipeline complete — all eight stops, one project.")
# go ship something real.

out = runner(inputs=sample)